# Verify, demonstrate, and preliminarily analyze M3

This notebook validates M3a-M3c for RQ2, demonstrates official tables and article-ready figures, and provides descriptive context for the M4-M5 contrast. M3 measures repository activity timing and authorship, not coordination friction, effort, productivity, or AI use.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import plotly.express as px
from IPython.display import Image, display
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'paper_v9').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
METRICS = ROOT / 'paper_v9' / 'data' / 'metrics'
FIGURES = ROOT / 'paper_v9' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
participation = pd.read_csv(METRICS / 'm3_author_activity_participation.csv', dtype={'Semestre': str})
concentration = pd.read_csv(METRICS / 'm3_author_concentration.csv', dtype={'Semestre': str})
rolling = pd.read_csv(METRICS / 'm3_activity_rolling_7day.csv', dtype={'Semestre': str})
pooled_rolling = pd.read_csv(METRICS / 'm3_activity_rolling_7day_pooled.csv')
metadata = json.loads((METRICS / 'm3_author_activity_dynamics.metadata.json').read_text())


In [ ]:
assert len(participation) == 14
assert len(concentration) == 28
assert len(rolling) == 406
assert len(pooled_rolling) == 29
assert 'Semestre' not in pooled_rolling.columns
assert set(pooled_rolling['window_end_day']) == set(range(-21, 8))
assert pooled_rolling['team_semester_n'].eq(14).all()
assert set(participation['Semestre'].astype(str)) == {'2025.2', '2026.1'}
assert participation[['pre_share', 'post_share', 'final_window_share']].apply(lambda column: column.dropna().between(0, 1).all()).all()
assert set(concentration['phase']) == {'pre', 'post'}
assert set(rolling['window_end_day']) == set(range(-21, 8))
no_activity = concentration['measurement_status'].eq('no_observed_activity')
assert concentration.loc[no_activity, ['max_author_share', 'author_gini']].isna().all().all()
assert not set(concentration.columns).intersection({'author_email', 'author_name'})
assert metadata['gate_status'] == 'approved'
assert metadata['gate_approved_on'] == '2026-09-24'
legacy = pd.read_csv(ROOT / 'paper_v8' / 'data' / 'm3_author_concentration_density.csv', dtype={'Semestre': str})
assert len(legacy) == 42
print('approved M3 schemas, pooled grain, ranges, privacy, and legacy availability: PASS')


In [ ]:
participation['team_semester'] = participation['Semestre'] + '/' + participation['ID_Equipe']
phase_figure = px.bar(participation, x='team_semester', y=['pre_share', 'post_share'], title='M3a final-phase commit share by team-semester', labels={'value': 'Commit share', 'variable': 'Phase'})
phase_figure.update_yaxes(range=[0, 1], tickformat='.0%')
rolling_summary = rolling.groupby(['Semestre', 'window_end_day'], as_index=False).agg(commit_n=('commit_n', 'sum'), active_team_n=('commit_n', lambda values: values.gt(0).sum()))
rolling_figure = px.line(rolling_summary, x='window_end_day', y='commit_n', color='Semestre', markers=True, title='M3c rolling seven-day activity by semester')
rolling_figure.add_vline(x=0, line_dash='dash')
pooled_figure = px.line(pooled_rolling, x='window_end_day', y='total_commit_n', markers=True, title='M3c rolling seven-day activity pooled across semesters', labels={'total_commit_n': 'Commits across all team-semesters', 'window_end_day': 'Window end relative to T3 anchor'})
pooled_figure.add_vline(x=0, line_dash='dash')
for figure, stem in ((phase_figure, 'm3_final_phase_commit_share'), (rolling_figure, 'm3_rolling_activity_7day'), (pooled_figure, 'm3_rolling_activity_7day_pooled')):
    figure.write_html(METRICS / f'{stem}.html', include_plotlyjs='cdn')
    for extension in ('pdf', 'svg', 'png'):
        figure.write_image(FIGURES / f'{stem}.{extension}', scale=2 if extension == 'png' else 1)
for stem in ('m3_final_phase_commit_share', 'm3_rolling_activity_7day', 'm3_rolling_activity_7day_pooled'):
    assert all((FIGURES / f'{stem}.{extension}').is_file() for extension in ('pdf', 'svg', 'png'))
print('M3 stratified and semester-independent article-ready figures generated: PASS')


In [ ]:
display(participation[['Semestre', 'ID_Equipe', 'total_commit_n', 'pre_commit_n', 'post_commit_n', 'final_window_share', 'small_denominator_lt_10']].sort_values(['Semestre', 'ID_Equipe']))
display(concentration[['Semestre', 'ID_Equipe', 'phase', 'commit_n', 'active_days', 'author_n', 'max_author_share', 'measurement_status']].head(12))
display(pooled_rolling[pooled_rolling['window_end_day'].isin([-7, 0, 1, 7])])
for path in (FIGURES / 'm3_final_phase_commit_share.png', FIGURES / 'm3_rolling_activity_7day.png', FIGURES / 'm3_rolling_activity_7day_pooled.png'):
    assert path.is_file() and path.stat().st_size > 0
    display(Image(filename=str(path), width=900))
print('M3 artifact demo: stratified and pooled tables and figures displayed')


## Preliminary RQ2 reading

M3a identifies temporal concentration of repository activity around the evaluator-vote anchor. M3b describes whether the final activity is concentrated among authors, and M3c shows whether activity peaks near the anchor. These observations provide repository context for the RQ2 contrast with M4 clean-change magnitude and M5 qualitative coordination evidence. They do not establish friction, causality, effort, productivity, or planning quality.

In [ ]:
post_activity = participation['post_commit_n'].gt(0)
summary = pd.DataFrame([{
    'team_semesters': len(participation),
    'post_active_team_semesters': int(post_activity.sum()),
    'median_final_window_share': participation['final_window_share'].median(),
    'pooled_peak_window_end_day': int(pooled_rolling.loc[pooled_rolling['total_commit_n'].idxmax(), 'window_end_day']),
    'pooled_peak_commit_n': int(pooled_rolling['total_commit_n'].max()),
    'small_denominator_team_semesters': int(participation['small_denominator_lt_10'].sum()),
    'rolling_window_rows': len(rolling),
}])
display(summary.round(3))
assert metadata['rq'] == 'RQ2'
assert metadata['coverage']['team_semesters'] == 14
assert summary.loc[0, 'pooled_peak_window_end_day'] in {-1, 0, 1}
print('preliminary RQ2 reading: pooled rolling activity describes common temporal shape across semesters; stratified series remain necessary for cohort differences')
